In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
label_dir = "/content/drive/MyDrive/UXO_Detector/labels/train"

classes = set()

import os

for file in os.listdir(label_dir):
    with open(os.path.join(label_dir, file)) as f:
        for line in f:
            if line.strip():
                cls = int(line.split()[0])
                classes.add(cls)

print(sorted(classes))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [8]:
!pip install -q ultralytics

import torch
from ultralytics import YOLO

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4


In [9]:
DATA_PATH = "/content/drive/MyDrive/UXO_Detector/data.yaml"
PROJECT_PATH = "/content/drive/MyDrive/UXO_Detector/runs"

In [11]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=DATA_PATH,

    epochs=100,
    imgsz=640,
    batch=16,

    device=0,
    workers=2,

    optimizer="AdamW",
    lr0=0.001,

    patience=15,

    cache=True,
    amp=True,
    pretrained=True,

    project=PROJECT_PATH,
    name="yolo_colab_run",

    verbose=True
)

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/UXO_Detector/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_colab_run-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overla

In [12]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,584,492 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 233.5±138.2 MB/s, size: 1078.4 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1Hl1rHkuzYz5F7bRUmxK3WKs4JrdDbaDA/UXO_Detector/labels/valid.cache... 486 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 486/486 145.6Mit/s 0.0s
val: /content/drive/.shortcut-targets-by-id/1Hl1rHkuzYz5F7bRUmxK3WKs4JrdDbaDA/UXO_Detector/images/valid/UXOs_2552.jpg: 1 duplicate labels removed
val: /content/drive/.shortcut-targets-by-id/1Hl1rHkuzYz5F7bRUmxK3WKs4JrdDbaDA/UXO_Detector/images/valid/UXOs_2956.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 1.5it/s 20.8s
                   all        486       1766      0.634      0.324       0.37      0.297
    AntiSubmarine Bomb          1     

In [14]:
model.predict(
    source="/content/drive/MyDrive/UXO_Detector/images/test",
    conf=0.25,

    imgsz=640,
    batch=4,
    stream=True,

    device=0,
    half=True,

    save=True,
    project=PROJECT_PATH,
    name="predictions",
    show_labels=True,
    show_conf=True,
    line_width=2,

    save_txt=False,
    save_conf=False,
    verbose=False
)

<generator object BasePredictor.stream_inference at 0x7f308c432b00>

In [15]:
model.export(
    format="tflite",

    imgsz=416,

    data=DATA_PATH,
    fraction=0.1,

    batch=1,
    workspace=2,

    optimize=True,
    verbose=False
)

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/.shortcut-targets-by-id/1Hl1rHkuzYz5F7bRUmxK3WKs4JrdDbaDA/UXO_Detector/runs/yolo_colab_run-4/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 16, 3549) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 208ms
Prepared 4 packages in 10.05s
Installed 4 packages in 468ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.25.1
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 11.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: sl

'/content/drive/.shortcut-targets-by-id/1Hl1rHkuzYz5F7bRUmxK3WKs4JrdDbaDA/UXO_Detector/runs/yolo_colab_run-4/weights/best_saved_model/best_float32.tflite'

**PLOTTING THE TRAINING VISUALIZATIONS TO CHECK MODEL PERFORMANCE**

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import os

RUN_PATH = "/content/drive/MyDrive/UXO_Detector/runs/yolo_colab_run"

csv_path = os.path.join(RUN_PATH, "results.csv")
df = pd.read_csv(csv_path)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/UXO_Detector/runs/yolo_colab_run/results.csv'

In [ ]:
# Loss curves
plt.figure()
plt.plot(df['train/box_loss'], label='Train Box Loss')
plt.plot(df['val/box_loss'], label='Val Box Loss')
plt.legend()
plt.title("Box Loss")
plt.savefig(os.path.join(RUN_PATH, "box_loss_curve.png"))
plt.close()


# mAP curves
plt.figure()
plt.plot(df['metrics/mAP50(B)'], label='mAP@0.5')
plt.plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
plt.legend()
plt.title("mAP Curve")
plt.savefig(os.path.join(RUN_PATH, "map_curve.png"))
plt.close()

In [ ]:
plt.figure()
plt.plot(df['metrics/precision(B)'], label='Precision')
plt.plot(df['metrics/recall(B)'], label='Recall')
plt.legend()
plt.title("Precision vs Recall")
plt.savefig(os.path.join(RUN_PATH, "precision_recall_curve.png"))
plt.close()

In [ ]:
df['F1'] = 2 * (df['metrics/precision(B)'] * df['metrics/recall(B)']) / \
           (df['metrics/precision(B)'] + df['metrics/recall(B)'] + 1e-6)

df[['epoch', 'F1']].to_csv(os.path.join(RUN_PATH, "f1_scores.csv"), index=False)

In [ ]:
metrics = model.val()

# Extract per-class metrics
results_dict = metrics.results_dict

report = pd.DataFrame({
    "Class": list(results_dict.keys()),
    "Value": list(results_dict.values())
})

report.to_csv(os.path.join(RUN_PATH, "detection_report.csv"), index=False)

In [ ]:
summary = {
    "Final mAP@0.5": df['metrics/mAP50(B)'].iloc[-1],
    "Final mAP@0.5:0.95": df['metrics/mAP50-95(B)'].iloc[-1],
    "Final Precision": df['metrics/precision(B)'].iloc[-1],
    "Final Recall": df['metrics/recall(B)'].iloc[-1],
    "Final F1": df['F1'].iloc[-1],
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(os.path.join(RUN_PATH, "final_summary.csv"), index=False)

In [ ]:
import os

img_dir = "/content/drive/MyDrive/UXO_Detector/images/valid/images"
lbl_dir = "/content/drive/MyDrive/UXO_Detector/labels/valid"

imgs = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
lbls = sorted([f.replace('.txt','') for f in os.listdir(lbl_dir)])

missing = [img for img in imgs if img.rsplit('.',1)[0] not in lbls]

print("Total images:", len(imgs))
print("Total labels:", len(lbls))
print("Images without labels:", len(missing))
print("Sample missing:", missing[:5])

In [18]:
import os
import shutil

BASE = "/content/drive/MyDrive/UXO_Detector/runs/yolo_colab_run-4/weights"
EXPORT_DIR = "/content/exported_model"

os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy TFLite model
shutil.copy(
    os.path.join(BASE, "best_saved_model", "best_float32.tflite"),
    os.path.join(EXPORT_DIR, "model.tflite")
)

# Create labels.txt
labels = [
    'AntiSubmarine Bomb', 'Aviation Bomb', 'Cartridge',
    'Cartridge Magazine', 'Fuse', 'Grenade',
    'LandMine', 'Mortar Bomb', 'Projectile',
    'RPG', 'Rocket', 'Sea Mine'
]

with open(os.path.join(EXPORT_DIR, "labels.txt"), "w") as f:
    for l in labels:
        f.write(l + "\n")

# Zip everything
shutil.make_archive("/content/uxo_detector_android", 'zip', EXPORT_DIR)

print("Download this file: /content/uxo_detector_android.zip")

Download this file: /content/uxo_detector_android.zip
